In [1]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import warnings

warnings.filterwarnings("ignore")

# --- 1. 路径设置 ---
# 确保路径指向 Code2 根目录
path = os.path.join(os.getcwd(), '..', '..')
sys.path.append(path)

from utils.common_utils import printlog, set_seed, make_dir
from models.LD_Net import LD_Net
from models.CNNRNNs import CNNRNNs
from trainTest.optimizers.get_optimizer import Optimizer
from trainTest.losses.get_loss import get_classification_loss
from trainTest.lr_schedulers.get_lr_scheduler import LrScheduler
from trainTest.metrics.metrics_utils import (
    get_accuracy, get_precision, get_recall, get_f1, get_specificity
)
from sklearn.metrics import roc_auc_score

# --- 3. 全局设置 ---
set_seed(seed=42)
print("UCI 依赖导入成功。")

UCI 依赖导入成功。


In [2]:
# 单元格 2
# === 1. 数据集与模型配置 ===
DATASET_NAME = "UCI"
MODEL_TYPE = 'SC-BiGRU_UCI' # 保持 BiGRU
DENOISE_METHOD = 'LD_Net_Online_Classification_UCI' # 结果保存在这里

C = 4
n_channels_check = 4

# === 2. 关键路径设置 ===

# (1) LD-Net 权重 (UCI 版本)
LD_NET_MODEL_PATH = os.path.join(
    os.getcwd(), 'checkpoints', 'LD_Net_UCI', 'ld_net_best_model.pth'
)

# (2) 数据集路径 (UCI)
base_data_path = os.path.join(path, "preProcessing")
DATA_DIR = os.path.join(
    base_data_path, "UCI_trainData", "Denoising_TrainSet_XY"
)

# === 3. 训练参数 (与 SIAT 保持一致的逻辑) ===
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 200
PATIENCE = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === 4. 检查与验证 ===
print(f"--- 任务: {DATASET_NAME} (固定 LD-Net + BiGRU) [5次重复实验] ---")
print(f"设备: {DEVICE}")
print(f"LD-Net 路径: {os.path.abspath(LD_NET_MODEL_PATH)}")
print(f"数据源路径: {os.path.abspath(DATA_DIR)}")

# === 5. 结果保存路径 ===
save_dir_root = os.path.join(path, 'results', DENOISE_METHOD)
save_dir_models_temp = os.path.join(save_dir_root, 'models') # 保持结构一致
make_dir(save_dir_root)
make_dir(save_dir_models_temp)

print(f"✅ 结果将保存到: {os.path.abspath(save_dir_root)}")

--- 任务: UCI (固定 LD-Net + BiGRU) ---
设备: cuda
LD-Net 路径: E:\基于肌电图的下肢运动模式识别\Code2\trainTest\LD_train Test\checkpoints\LD_Net_UCI\ld_net_best_model.pth
数据源路径: E:\基于肌电图的下肢运动模式识别\Code2\preProcessing\UCI_trainData\Denoising_TrainSet_XY
✅ 结果将保存到: E:\基于肌电图的下肢运动模式识别\Code2\results\LD_Net_Online_Classification_UCI


In [3]:
#单元格3
print("--- 正在加载 UCI 版本的 LD-Net 降噪模型 ---")
try:
    model_denoise = torch.load(LD_NET_MODEL_PATH, weights_only=False)
    model_denoise.to(DEVICE)

    # --- 关键: 冻结 LD-Net (方案1) ---
    model_denoise.eval()
    for p in model_denoise.parameters():
        p.requires_grad = False
    # -------------------------------

    print("✅ LD-Net (UCI) 加载成功并已冻结 (eval模式)。")
except Exception as e:
    print(f"❌ 加载 LD-Net 失败: {e}")
    print("请检查 'checkpoints/LD_Net_UCI/' 下是否有对应的 .pth 文件。")
    # 如果你没有训练 UCI 的 LD-Net，这里会报错。
    raise e

--- 正在加载 UCI 版本的 LD-Net 降噪模型 ---
✅ LD-Net (UCI) 加载成功并已冻结 (eval模式)。


In [4]:
#单元格4
import glob

class LdNetClassificationDataset(Dataset):
    def __init__(self, data_dir, subject_id):
        self.data_dir = data_dir
        self.subject_id = subject_id

        # UCI 文件名格式: "1N_XY_TrainData.npz"
        file_name_prefix = '_XY_TrainData.npz'
        search_path = os.path.join(self.data_dir, f"{self.subject_id}{file_name_prefix}")
        file_paths = glob.glob(search_path)

        if not file_paths:
            print(f"警告: 未找到 {self.subject_id} 的数据文件 ({search_path})")
            self.samples = np.empty((0, C, 128)) # 假设窗口为128
            self.labels_encoded = np.empty((0,))
            return

        try:
            data = np.load(file_paths[0])
            self.samples = data['noisy_X'].astype(np.float32)
            self.labels_encoded = data['sub_motion_label_encoded'].astype(np.int64)
            print(f"  已加载 {self.subject_id}: {self.samples.shape[0]} 个样本。")
        except Exception as e:
            print(f"加载 {file_paths[0]} 出错: {e}")
            self.samples = np.empty((0, C, 128))
            self.labels_encoded = np.empty((0,))

    def __len__(self):
        return self.samples.shape[0]

    def __getitem__(self, idx):
        return self.samples[idx], self.labels_encoded[idx]

print("✅ UCI 数据集类定义完毕。")

✅ UCI 数据集类定义完毕。


In [7]:
# 单元格 5
from sklearn.model_selection import train_test_split
from trainTest.metrics.get_test_metrics import PlotConfusionMatrix
import torch.optim as optim
import numpy as np
import torch.nn as nn
from trainTest.metrics.metrics_utils import (
    get_accuracy, get_precision, get_recall, get_f1, get_specificity
)
from sklearn.metrics import roc_auc_score
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from trainTest.early_stopping.early_stopping import EarlyStopping
from trainTest.optimizers.get_optimizer import Optimizer
from trainTest.losses.get_loss import get_classification_loss
from trainTest.lr_schedulers.get_lr_scheduler import LrScheduler
# ----------------------------------------------------
def _calc_cls_metrics(y_true, y_pred, y_scores=None):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    metrics = {
        'Accuracy': get_accuracy(y_true, y_pred), 'Precision': get_precision(y_true, y_pred),
        'Recall': get_recall(y_true, y_pred), 'F1': get_f1(y_true, y_pred),
        'Specificity': get_specificity(y_true, y_pred) * 100.0,
    }
    auc_pct = 0.0
    if y_scores is not None:
        if isinstance(y_scores, list):
            try: y_scores = np.asarray(y_scores)
            except Exception: y_scores = None
        if isinstance(y_scores, np.ndarray) and y_scores.ndim == 2 and y_scores.shape[1] > 1:
            try:
                auc = roc_auc_score(y_true, y_scores, multi_class='ovr', average='macro')
                auc_pct = round(auc * 100.0, 3)
            except Exception: auc_pct = 0.0
    metrics['AUC'] = auc_pct
    return metrics

# --- 形状规整器 (适配 C=4) ---
def _coerce_to_1xCxL(out, channels=4):
    if isinstance(out, (list, tuple)): t = out[0]
    else: t = out

    if t.dim() == 3 and t.size(1) == channels:
        return t.unsqueeze(1) # [B, C, L] -> [B, 1, C, L]
    elif t.dim() == 4 and t.size(1) == 1:
        return t
    else:
        raise RuntimeError(f"LD-Net 输出形状异常: {tuple(t.shape)}, 期望通道 {channels}")

def train_one_epoch(model_classifier, model_denoiser, train_loader, criterion, optimizer, device):
    model_classifier.train()
    # model_denoiser 是冻结的
    train_loss = 0.; total_num = 0
    y_true_all = []; y_pred_all = []

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        with torch.no_grad():
            out = model_denoiser(inputs)
            inputs_denoised = _coerce_to_1xCxL(out, channels=C) # C=4

        outputs = model_classifier(inputs_denoised)
        loss    = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        bs = labels.size(0)
        train_loss += loss.item() * bs; total_num += bs
        y_true_all.extend(labels.detach().cpu().numpy())
        y_pred_all.extend(torch.max(outputs, 1)[1].detach().cpu().numpy())

    return train_loss/total_num, _calc_cls_metrics(y_true_all, y_pred_all, None)

@torch.no_grad()
def test_one_epoch(model_classifier, model_denoiser, test_loader, criterion, device):
    model_classifier.eval()
    # model_denoiser 是冻结的
    test_loss = 0.; total_num = 0
    y_true_all = []; y_pred_all = []; y_scores_all = []

    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        out = model_denoiser(inputs)
        inputs_denoised = _coerce_to_1xCxL(out, channels=C) # C=4

        outputs = model_classifier(inputs_denoised)
        loss    = criterion(outputs, labels)

        bs = labels.size(0)
        test_loss += loss.item() * bs; total_num += bs
        y_true_all.extend(labels.detach().cpu().numpy())
        y_pred_all.extend(torch.max(outputs, 1)[1].detach().cpu().numpy())
        y_scores_all.extend(F.softmax(outputs, dim=1).detach().cpu().numpy())

    return test_loss/total_num, _calc_cls_metrics(y_true_all, y_pred_all, np.asarray(y_scores_all))

def train_test_subject(subject_id, data_dir, model_denoise, device, save_dir_models_temp,
                       model_type, epochs, batch_size, lr, patience, exp_tim):

    # 1. 加载数据
    dataset = LdNetClassificationDataset(data_dir=data_dir, subject_id=subject_id)
    if len(dataset) == 0: return None

    # 2. 划分 (使用 exp_tim 作为随机种子，实现 5 次重复实验)
    train_idx, test_idx = train_test_split(
        list(range(len(dataset))), test_size=0.2, random_state=exp_tim, stratify=dataset.labels_encoded
    )
    train_loader = DataLoader(torch.utils.data.Subset(dataset, train_idx), batch_size=batch_size, shuffle=True,  num_workers=0)
    test_loader  = DataLoader(torch.utils.data.Subset(dataset, test_idx),  batch_size=batch_size, shuffle=False, num_workers=0)

    # 3. 模型 (BiGRU)
    # CNNRNNs 会自动读取 common_params 中的 C=4
    model_classifier = CNNRNNs(rnn_type='BiGRU').to(device)

    # 4. 优化器 & Loss
    optimizer = Optimizer(model=model_classifier, optimizer_type='Adam', lr=lr).get_optimizer()
    criterion = get_classification_loss()

    # 5. Scheduler & ES
    scheduler_params = {'ReduceLROnPlateau': {'mode':'min','factor':0.1,'patience':5,'threshold':1e-4,'min_lr':1e-8}}
    scheduler = LrScheduler(optimizer=optimizer, scheduler_type='ReduceLROnPlateau',
                            params=scheduler_params, max_epoch=epochs).get_scheduler()

    save_dir_subj = os.path.join(save_dir_models_temp, subject_id)
    make_dir(save_dir_subj)
    best_model_path = os.path.join(save_dir_subj, f'best_model_exp{exp_tim}.pth')
    early_stopping = EarlyStopping(patience=patience, verbose=True, path=best_model_path)

    # 6. 训练循环
    for epoch in range(1, epochs+1):
        train_loss, train_metrics = train_one_epoch(
            model_classifier, model_denoise, train_loader, criterion, optimizer, device
        )
        val_loss, val_metrics = test_one_epoch(
            model_classifier, model_denoise, test_loader, criterion, device
        )

        scheduler.step(val_loss)
        early_stopping(val_loss, model_classifier) # 只保存分类器

        if (epoch % 20 == 0):
             print(f"{subject_id} | Exp {exp_tim} | Ep {epoch:03d} | Val Acc: {val_metrics['Accuracy']:.2f}%")

        if early_stopping.early_stop:
            break

    # 7. 最终测试
    best_classifier = torch.load(best_model_path, weights_only=False).to(device)
    _, test_metrics_final = test_one_epoch(
        best_classifier, model_denoise, test_loader, criterion, device
    )

    # 8. 返回
    test_metrics_final['Subject'] = subject_id
    return test_metrics_final

print("✅ UCI 训练函数 (5次重复实验版) 定义完毕。")

✅ UCI 训练函数 (5次重复实验版) 定义完毕。


In [8]:
# 单元格 6
import pandas as pd
import numpy as np

# --- 1. 定义受试者 (1N-5N, 7A-11A) ---
subjects_list = [f"{i}N" for i in range(1, 6)] + [f"{i}A" for i in range(7, 12)]
print(f"目标受试者: {subjects_list}")

# --- 2. 初始化 ---
metrics_all_subjects = []
# 复用 SIAT 的 K 值
K_of_repeated_experiments = 5

printlog(f"--- 开始 UCI 数据集 5次重复实验 (Fixed LD-Net + BiGRU) ---", time=True, line_break=True)

# --- 3. 双重循环 ---
for subject in subjects_list:
    for exp_tim in range(1, K_of_repeated_experiments + 1):

        print(f"正在处理: {subject} | 实验: {exp_tim}/{K_of_repeated_experiments} ...")

        # 调用训练函数
        test_metrics = train_test_subject(
            subject_id=subject,
            data_dir=DATA_DIR,
            model_denoise=model_denoise,
            device=DEVICE,
            save_dir_models_temp=save_dir_models_temp,
            model_type=MODEL_TYPE,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
            patience=PATIENCE,
            exp_tim=exp_tim
        )

        if test_metrics:
            test_metrics['Experiment'] = exp_tim
            metrics_all_subjects.append(test_metrics)
            print(f"  >>> {subject} 实验 {exp_tim} 完成。Acc: {test_metrics['Accuracy']:.2f}%")

printlog(f"--- 所有 UCI 受试者处理完毕 ---", time=True, line_break=True)

# --- 4. 保存结果 (完全一致的 SIAT 格式) ---
if metrics_all_subjects:
    # 4.1. 保存【所有运行】的详细原始数据
    df_summary = pd.DataFrame(metrics_all_subjects)
    save_path_summary = os.path.join(save_dir_root, 'summary_by_subject.csv')
    df_summary.to_csv(save_path_summary, index=False, float_format='%.3f')
    print(f"✅ 详细CSV已保存到: {save_path_summary}")

    # 4.2. 计算【每个受试者的平均值】(数字)
    df_avg_by_subject = df_summary.groupby('Subject').mean(numeric_only=True)
    # 计算【每个受试者的标准差】(数字)
    df_std_by_subject = df_summary.groupby('Subject').std(numeric_only=True)

    # 4.3. 创建【受试者行】的格式化 (mean+std) DataFrame
    df_formatted_subjects = pd.DataFrame(index=df_avg_by_subject.index)
    metrics_cols = [col for col in df_avg_by_subject.columns if col != 'Experiment']

    for col in metrics_cols:
        df_formatted_subjects[col] = df_avg_by_subject[col].map('{:.4f}'.format) + \
                                     "+" + \
                                     df_std_by_subject[col].map('{:.4f}'.format)

    # 4.4. 计算【总平均值】和【总标准差】
    df_total_avg = df_avg_by_subject[metrics_cols].mean().to_frame().T
    df_total_std = df_avg_by_subject[metrics_cols].std().to_frame().T

    # 4.5. 格式化总平均
    df_total_avg_formatted = df_total_avg.applymap(lambda x: f"{x:.4f}")
    df_total_std_formatted = df_total_std.applymap(lambda x: f"{x:.4f}")
    df_total_avg_formatted.index = ['mean']
    df_total_std_formatted.index = ['std']

    # 4.6. 拼接并保存
    df_final_report = pd.concat([df_formatted_subjects, df_total_avg_formatted, df_total_std_formatted])

    save_path_final_report = os.path.join(save_dir_root, 'all_metrics_averaged_results.csv')
    df_final_report.to_csv(save_path_final_report, index=True)
    print(f"✅ 【最终报告 (mean+std)】已保存到: {save_path_final_report}")

else:
    print("❌ 未处理任何受试者，请检查 DATA_DIR 路径和文件名。")

目标受试者: ['1N', '2N', '3N', '4N', '5N', '7A', '8A', '9A', '10A', '11A']

============================================================2025-11-12 14:16:08
--- 开始 UCI 数据集 5次重复实验 (Fixed LD-Net + BiGRU) ---...

正在处理: 1N | 实验: 1/5 ...
  已加载 1N: 364 个样本。
Using Adam optimizer, initial learning rate:  0.001
Validation loss decreased (inf --> 1.093940).  Saving model ...
Validation loss decreased (1.093940 --> 1.080245).  Saving model ...
Validation loss decreased (1.080245 --> 1.038932).  Saving model ...
Validation loss decreased (1.038932 --> 0.961582).  Saving model ...
Validation loss decreased (0.961582 --> 0.877263).  Saving model ...
Validation loss decreased (0.877263 --> 0.622695).  Saving model ...
Validation loss decreased (0.622695 --> 0.573730).  Saving model ...
Validation loss decreased (0.573730 --> 0.442037).  Saving model ...
Validation loss decreased (0.442037 --> 0.408036).  Saving model ...
Validation loss decreased (0.408036 --> 0.354394).  Saving model ...
Validation loss d